# 🎭 Free Talking Photo & Lip-Sync Video Studio
### ১০০% ফ্রি গুগল কোলাব GPU ব্যবহার করে যেকোনো ছবি ও অডিও থেকে নিখুঁত লিপ-সিঙ্ক ভিডিও তৈরি করুন

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaalikolom-ux/video-project/blob/main/colab_talking_photo_lipsync.ipynb)

---
### 📌 ব্যবহারের নিয়ম:
1. উপরে **Runtime -> Change runtime type**-এ গিয়ে **T4 GPU** সিলেক্ট করা আছে কিনা দেখে নিন (ফ্রি)।
2. নিচের **ধাপ ১** রান করুন (প্রয়োজনীয় মডেল ও কোড ডাউনলোড হবে - প্রায় ২ মিনিট সময় লাগবে)।
3. **ধাপ ২** রান করুন (আপনার ছবি ও আবৃত্তির অডিও লোড হবে)।
4. **ধাপ ৩** রান করুন (সম্পূর্ণ নিখুঁত লিপ-সিঙ্ক ভিডিও রেন্ডার হবে এবং স্বয়ংক্রিয়ভাবে ডাউনলোড হয়ে যাবে)।

## ⚙️ ধাপ ১: পরিবেশ ও এআই মডেল সেটআপ (Setup Environment & GPU)

In [ ]:
# Check GPU availability
!nvidia-smi

# Clone SadTalker repository
!git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker
%cd /content/SadTalker

# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q facexlib gfpgan yacs

# Download pre-trained weights
!mkdir -p checkpoints
!mkdir -p gfpgan/weights
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar -O ./checkpoints/mapping_00109-model.pth.tar
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar -O ./checkpoints/mapping_00229-model.pth.tar
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors -O ./checkpoints/SadTalker_V0.0.2_256.safetensors
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors -O ./checkpoints/SadTalker_V0.0.2_512.safetensors
!wget -nc https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth -O ./gfpgan/weights/alignment_WFLW_4HG.pth
!wget -nc https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth -O ./gfpgan/weights/detection_Resnet50_Final.pth
!wget -nc https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -O ./gfpgan/weights/GFPGANv1.4.pth
!wget -nc https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth -O ./gfpgan/weights/parsing_parsenet.pth

print("\n✅ মডেল ও লাইব্রেরি সফলভাবে সেটআপ হয়েছে!")

## 📸 ধাপ ২: ছবি ও অডিও লোড করুন (Load Image & Poem Audio)
নিচের কোডটি রান করলে আপনার GitHub রিপোজিটরি থেকে আজকের কবিতা ও ছবিটি সরাসরি লোড হয়ে যাবে।
(আপনি চাইলে নিজের নতুন কোনো ছবি বা অডিও-ও আপলোড করতে পারবেন)।

In [ ]:
import os
import urllib.request
from google.colab import files

# Option A: Auto-fetch your current poem portrait & 66s recitation audio
img_url = "https://raw.githubusercontent.com/kaalikolom-ux/video-project/main/assets/poem_portrait.jpg"
audio_url = "https://raw.githubusercontent.com/kaalikolom-ux/video-project/main/assets/poem_recitation.mp3"

urllib.request.urlretrieve(img_url, "/content/source_image.jpg")
urllib.request.urlretrieve(audio_url, "/content/driven_audio.mp3")

print("✅ আপনার ছবি (source_image.jpg) এবং কবিতার আবৃত্তি (driven_audio.mp3) সফলভাবে লোড হয়েছে!")
print("💡 আপনি যদি নতুন কোনো ছবি বা অডিও দিতে চান, তবে নিচের কোডটি আন-কমেন্ট করে রান করতে পারেন:")
# uploaded = files.upload() # যেকোনো নতুন ফাইল আপলোড করতে এটি ব্যবহার করুন

## 🎬 ধাপ ৩: সম্পূর্ণ লিপ-সিঙ্ক ভিডিও তৈরি ও ডাউনলোড (Render & Auto-Download)
এই সেলটি রান করুন। এটি অডিওর প্রতিটি শব্দের সাথে ঠোঁটের নড়াচড়া (Lip-Sync), চোখের পলক ও মুখের অভিব্যক্তি সিঙ্ক করে সম্পূর্ণ এইচডি ভিডিও তৈরি করবে এবং ভিডিওটি আপনার কম্পিউটারে ডাউনলোড করে দেবে।

In [ ]:
import glob
from google.colab import files
from IPython.display import HTML
from base64 import b64encode

%cd /content/SadTalker
!rm -rf /content/results

# Run inference with face enhancer (GFPGAN) for high-definition quality
!python inference.py \
    --driven_audio /content/driven_audio.mp3 \
    --source_image /content/source_image.jpg \
    --result_dir /content/results \
    --still \
    --preprocess full \
    --enhancer gfpgan

# Find generated video file
video_files = glob.glob("/content/results/*.mp4")
if video_files:
    latest_video = max(video_files, key=os.path.getctime)
    print(f"\n🎉 ভিডিও সফলভাবে তৈরি হয়েছে: {latest_video}")
    
    # Preview video in notebook
    mp4 = open(latest_video, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'''
    <video width=480 controls autoplay>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    
    # Trigger auto-download to user's PC
    print("⬇️ কম্পিউটারে ডাউনলোড শুরু হচ্ছে...")
    files.download(latest_video)
else:
    print("❌ কোনো ভিডিও খুঁজে পাওয়া যায়নি। দয়া করে এরর লগ চেক করুন।")